# DAAN 545 — Outlier detection on delay profiles

**Goal:** Flag carrier–airport–months whose delay profile is unusual, using three definitions on the same rows.

**Input:** `data/processed/airline_delay_cause_with_airports.csv` (BTS delays + OpenFlights geography + hub fields). FAA staffing is not an input.

**Approach:**
1. Reuse the clustering population and the six profile features. Standardize for the distance methods.
2. Flag **Tukey IQR** tails on the two rates.
3. Flag isolated profiles with **k-NN distance** and an **isolation forest**.
4. Compare the flags. A consensus outlier is a month flagged by at least two methods.

Write-up language and the numbers from the saved run: `docs/notebook_notes.md`.


## 0. Setup

Helpers live in `src/outliers.py`. The row filter and the feature list come from `src/clustering.py`, so this notebook scores the same months the cluster fit used.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.outliers import run_outlier_detection, save_outlier_figures, score_table

DATA_PROCESSED = ROOT / "data" / "processed"
FIG_DIR = ROOT / "reports" / "figures"
print(f"Project root: {ROOT}")


## 1. Three definitions of an outlier

Each row is still one carrier–airport–month. Months with `arr_flights` below the 10th percentile stay out, same as clustering. A handful of flights can make a rate swing from 0% to 100%.

Features, after `StandardScaler` for k-NN and the forest:

- `pct_delayed`, `pct_cancelled`
- share of delay minutes: carrier, weather, NAS, late aircraft

Security’s share stays out when its standard deviation is below the clustering cutoff. We **flag** rows. We do not drop them.

| Method | A month is an outlier when |
|---|---|
| **IQR rates** | `pct_delayed` or `pct_cancelled` is outside the Tukey fence on this population (Q1 − 1.5×IQR, Q3 + 1.5×IQR). Fences use the raw rates, not the scaled columns. |
| **k-NN distance** | The mean Euclidean distance to the 15 nearest other months, in the scaled space, is above the upper Tukey fence of that distance. |
| **Isolation forest** | `decision_function` < 0. The forest uses 200 trees, `contamination=0.05`, and `random_state=42`, so about 5% of months are marked. |

**Consensus** means at least two of those three flags. `rate_only` is IQR without either distance method. `shape_only` is k-NN or the forest without IQR. `rate_and_shape` is both.


In [ ]:
panel = pd.read_csv(DATA_PROCESSED / "airline_delay_cause_with_airports.csv")
result = run_outlier_detection(panel)

print("Rows in / scored / dropped:",
      result["meta"]["n_rows_in"],
      result["meta"]["n_rows_clustered"],
      result["meta"]["n_rows_dropped"])
print(f"Flight cutoff (10th percentile): {result['meta']['min_arr_flights']:.0f}")
print("Features:", result["features"])
print(f"PC1–PC2 variance: {result['pca']['variance_first_two']:.1%}")
print("Delay fence:", result["meta"]["iqr_delay_fence"])
print("Cancel fence:", result["meta"]["iqr_cancel_fence"])
print(
    "IQR counts — delay / cancel / both rates:",
    result["meta"]["n_iqr_delay"],
    result["meta"]["n_iqr_cancel"],
    result["meta"]["n_iqr_both_rates"],
)
print(
    f"Isolation forest with contamination='auto' would flag "
    f"{result['meta']['if_auto_n_outliers']:,} "
    f"({result['meta']['if_auto_pct']}%). The fit below uses contamination=0.05."
)
display(result["summary"])
print("k-NN sensitivity (same upper-fence rule at other k):")
display(result["knn_sweep"])
print("Logical checks on the scored months:")
display(result["logical_summary"])
print(
    "Logical flags that are also an outlier:",
    result["meta"]["n_logical_flags_among_outliers"],
    "of",
    result["meta"]["n_logical_flags"],
)


## 2. Where the methods agree

IQR answers whether a **rate** is in the tail. k-NN and the forest answer whether the **profile** is isolated. A month can be one without being the other.


In [ ]:
display(result["agreement"])
print("Pattern counts:")
display(result["frame"]["outlier_pattern"].value_counts().rename("n_rows").to_frame())


## 3. What the flagged months look like

Means are inside the group. Slot % is the share of rows at JFK, LGA, EWR, or DCA. Those airports were not features. The example table is the extreme end of each score, not a random sample of flags.


In [ ]:
display(result["profiles"])
display(result["examples"].round(3))


## 4. Figures and exports

The PCA scatter is a 2D view. k-NN and the forest were fit in the six-feature space, not in that plane. PC1–PC2 are under 70% of the variance.


In [ ]:
paths = save_outlier_figures(result, FIG_DIR)
for path in paths:
    print(path)

score_table(result).to_csv(DATA_PROCESSED / "outlier_scores.csv", index=False)
result["summary"].to_csv(DATA_PROCESSED / "outlier_method_summary.csv", index=False)
result["agreement"].to_csv(DATA_PROCESSED / "outlier_agreement.csv", index=False)
result["profiles"].to_csv(DATA_PROCESSED / "outlier_profiles.csv", index=False)
result["knn_sweep"].to_csv(DATA_PROCESSED / "outlier_knn_sweep.csv", index=False)
result["monthly"].to_csv(DATA_PROCESSED / "outlier_monthly_counts.csv", index=False)
result["examples"].to_csv(DATA_PROCESSED / "outlier_examples.csv", index=False)
print("Saved outlier tables under data/processed/")
